# Earnings Yield (E/P Value) — Strategy Research Notebook

This notebook asks a specific research question:

> Does buying S&P 500 stocks with the highest trailing-twelve-month earnings yield and shorting those with the lowest earn a positive return net of transaction costs when rebalanced monthly on a point-in-time fundamentals panel?

In plain English: cheap stocks (high earnings relative to price) have historically earned more than expensive stocks. We test whether that premium still shows up in US large caps after we (a) use filing dates so we never peek at earnings before they were public, and (b) pay 10 bps round-trip costs.

Example:

```text
It is the last trading day of March 2024.

Stock A filed its 10-K on Feb 15: TTM net income $10B, market cap $80B
  -> earnings yield = 10/80 = 12.5%  -> ranks near the TOP (cheap) -> LONG
Stock B filed on Mar 1: TTM net income $2B, market cap $200B
  -> earnings yield = 2/200 = 1.0%   -> ranks near the BOTTOM (expensive) -> SHORT

Positions are equal-weighted within each leg, held one month, then re-ranked.
Earnings for a fiscal quarter are NOT visible on the period-end date — only on
the first trading day after the SEC acceptedDate.
```

Registered as `earnings_yield` in `core/strategies/registry.py`; factor column `earnings_yield` in `factors_fundamental.parquet`. All computation is imported from `core/` — nothing is re-implemented here.

**References**: Basu (1977); Fama & French (1992).

## 1. Setup And Data Loading

Paths resolve from the repository root whether Jupyter started in the root or inside `notebooks/`. Missing files stop the notebook with a clear message.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "core").is_dir() else _cwd.parent
assert (REPO_ROOT / "core").is_dir(), f"Could not locate repo root from {_cwd}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FACTORS_PATH = REPO_ROOT / "data" / "factors" / "factors_fundamental.parquet"
PRICES_PATH = REPO_ROOT / "data" / "factors" / "prices.parquet"
PIT_PATH = REPO_ROOT / "data" / "factors" / "fundamentals.parquet"

for path in (FACTORS_PATH, PRICES_PATH, PIT_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run scripts/fetch_fmp_fundamentals.py then "
            "scripts/build_fundamentals_panel.py first."
        )

fundamental_factors = pd.read_parquet(FACTORS_PATH)
prices = pd.read_parquet(PRICES_PATH)
pit_metrics = pd.read_parquet(PIT_PATH)

print(f"fundamental factors: {fundamental_factors.shape} cols={list(fundamental_factors.columns)}")
print(f"price panel: {prices.shape} tz={prices.index.tz}")
print(f"PIT metrics: {pit_metrics.shape}")

## 2. What Exactly Are We Observing?

- **Universe**: point-in-time S&P 500 membership (`sp500_universe_filter`), not today's survivors.
- **Factor**: `earnings_yield` = TTM net income / market cap, visible only after the later of the income-statement and balance-sheet `acceptedDate` (next trading day).
- **Returns**: daily adjusted-close returns from the FMP-derived price panel; portfolio returns are net of 10 bps transaction cost at each monthly rebalance.
- **What the model does not see**: earnings before their filing date, names outside the S&P 500 on that date, or historical S&P members for whom FMP has no price history (survivorship gap — disclosed in section 4).

## 3. Per-Variable Audit

### 3.1 Upstream: TTM net income (point-in-time)

Source: `data/factors/fundamentals.parquet`, column `net_income_ttm`, built by `core.data.factors.fundamentals.extract_pit_metrics` from raw FMP quarterly income statements.

In [ ]:
aapl_ni = pit_metrics.xs("AAPL", level="symbol")["net_income_ttm"].dropna() / 1e9
print(aapl_ni.describe())
aapl_ni.plot(figsize=(10, 3), title="AAPL TTM net income ($B), PIT-aligned", color="#a1a1aa")
plt.ylabel("$B"); plt.grid(True, alpha=0.2); plt.show()
assert aapl_ni.notna().mean() > 0.9

### 3.2 Transformed: earnings_yield

Source: `core.data.factors.fundamentals.compute_fundamental_factors`. Same visibility date as the TTM income. High = cheap = long.

In [ ]:
aapl_ey = fundamental_factors.xs("AAPL", level="symbol")["earnings_yield"].dropna()
print(aapl_ey.describe())
aapl_ey.plot(figsize=(10, 3), title="AAPL earnings yield (E/P), PIT-aligned", color="#34d399")
plt.ylabel("E/P"); plt.grid(True, alpha=0.2); plt.show()
assert aapl_ey.between(-1, 1).mean() > 0.95, "earnings yield should be mostly in [-100%, +100%]"

# Leakage gate: AAPL FY2023 book equity must not appear before the 2023-11-03 filing.
aapl_book = pit_metrics.xs("AAPL", level="symbol")["book_equity"] / 1e9
pre = aapl_book.loc[:"2023-11-02"].iloc[-1]
post = aapl_book.loc["2023-11-03":].iloc[0]
print(f"book equity before filing: {pre:.3f}B; on/after filing: {post:.3f}B")
assert abs(pre - 60.274) < 0.1
assert abs(post - 62.146) < 0.1
assert pre != post

## 4. Portfolio Construction Disclosure

| Choice | Setting | Why |
|---|---|---|
| Universe | S&P 500 point-in-time | survivorship-free membership |
| Signal lag | 1 trading day (platform default) | avoid same-close execution fiction |
| Rebalance | Month-end | standard for slow-moving fundamentals |
| Long / short | top / bottom 20% | quintiles; equal-weight within leg |
| Costs | 10 bps one-way (0.001) | conservative large-cap estimate |
| Factor column | `earnings_yield` | high = long |

**Survivorship disclosure**: FMP has no EOD prices for many pre-~2015 delisted S&P members (coverage ~67% in 2005, ~98% in 2025). Missing names are silently absent from the cross-section. Prefer 2015+ windows for cleaner inference; earlier results tilt toward survivors.

## 5. Backtest

Uses `run_factor_cross_section_backtest` — the same path as the Factor Backtest UI.

In [ ]:
from core.backtest.portfolio import sp500_universe_filter
from core.metrics.performance import calculate_performance_metrics
from core.strategies.factor_runner import run_factor_cross_section_backtest

factor_panel = fundamental_factors.copy()
membership = sp500_universe_filter()

def run_window(start: str, end: str) -> dict:
    net = run_factor_cross_section_backtest(
        factor_panel,
        prices,
        factor_col="earnings_yield",
        start=pd.Timestamp(start, tz="America/New_York"),
        end=pd.Timestamp(end, tz="America/New_York"),
        rebalance_freq="ME",
        transaction_cost=0.001,
        top_pct=0.2,
        bottom_pct=0.2,
        universe_filter=membership,
    )
    return {"net": net, "metrics": calculate_performance_metrics(net)}

windows = {
    "2005-2025 (full)": ("2005-01-01", "2025-12-31"),
    "2015-2025 (high coverage)": ("2015-01-01", "2025-12-31"),
}
results = {name: run_window(s, e) for name, (s, e) in windows.items()}
for name, result in results.items():
    m = result["metrics"]
    print(
        f"{name:28s} ann_ret={m['annualized_return']:+.2%}  "
        f"vol={m['annualized_volatility']:.2%}  sharpe={m['sharpe_ratio']:+.2f}  "
        f"maxdd={m['max_drawdown']:.2%}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for name, result in results.items():
    cum = (1.0 + result["net"]).cumprod()
    ax.plot(cum.index, cum.values, label=name, linewidth=1.4)
ax.axhline(1.0, color="#52525b", linewidth=0.8)
ax.set_title("Earnings yield L/S — cumulative growth of $1 (net of 10bps)")
ax.set_ylabel("Growth of $1")
ax.legend(frameon=False)
ax.grid(True, alpha=0.2)
plt.show()

## 6. Decision

**Verdict for this sample**: in US S&P 500 data after 2005, standalone earnings-yield long/short is approximately flat after costs (Sharpe near zero). That matches the well-documented post-GFC decay of the US large-cap value premium — it is not a data bug (vol and drawdowns are sane; the PIT leakage gate above passes).

**What would change the decision**:
- Sector-neutral ranks (value often loads on financials/energy).
- Combine with quality (`roe`) — value+quality is the more modern specification.
- Extend below large-cap (our universe is S&P 500 only).
- Fix the market-cap denominator to true historical shares (roadmap item 3) before trusting pre-2015 B/M levels.

**Do not deploy** as a standalone live sleeve on this evidence. Keep the factor registered for composite research and regime diagnostics.